# Apparel Material Deduplication System
## Pipeline Step 3: Unique Master Materials (per category)

Reads each `data/04_curated/curated_<slug>.csv` **separately** and writes `data/05_unique/unique_master_<slug>.csv` with exactly one row per `new_master_code` and only these columns:
`new_master_code`, `norm_material_name`, `Material name`, `Category`, `cluster_id`, `cluster_record_count`.

Categories are never merged. `Category` is not stored in the curated files, so it is read from the matching `data/02_by_category/category_<slug>.csv` (falling back to the slug name).

In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "notebook"] else CURRENT_DIR
PARTITION_DIR = PROJECT_ROOT / "data" / "02_by_category"
CURATED_DIR = PROJECT_ROOT / "data" / "04_curated"
UNIQUE_DIR = PROJECT_ROOT / "data" / "05_unique"
UNIQUE_DIR.mkdir(parents=True, exist_ok=True)

curated_files = sorted(CURATED_DIR.glob("curated_*.csv"))
if not curated_files:
    raise FileNotFoundError(f"No curated_*.csv files in {CURATED_DIR}. Run notebook 02 first.")
print(f"[SETUP] Found {len(curated_files)} curated files in {CURATED_DIR}")

In [ ]:
FINAL_COLS = ["new_master_code", "norm_material_name", "Material name", "Category", "cluster_id", "cluster_record_count"]

def mode_value(s: pd.Series) -> str:
    """Most frequent non-empty value (ties -> first seen)."""
    s = s[s.str.strip() != ""]
    return s.value_counts(sort=True).index[0] if len(s) else ""

def category_for(slug: str) -> str:
    """Category label from the partition file, falling back to the slug."""
    part_path = PARTITION_DIR / f"category_{slug}.csv"
    if part_path.exists():
        cats = pd.read_csv(part_path, usecols=["Category"], dtype=str, keep_default_na=False)["Category"]
        label = mode_value(cats)
        if label:
            return label
    return slug.replace("_", " ").title()

def build_unique_master(slug: str):
    cur = pd.read_csv(CURATED_DIR / f"curated_{slug}.csv", dtype=str, keep_default_na=False)
    cur["_sim"] = cur["similarity_score"].astype(float)

    def canonical(df):
        # canonical record: similarity_score == 1.0 (else all rows), most frequent normalized name, then most frequent display name
        top = df[df["_sim"] == 1.0]
        top = top if len(top) else df
        norm = mode_value(top["norm_material_name"])
        name = mode_value(top.loc[top["norm_material_name"] == norm, "Material name"])
        return pd.Series({"norm_material_name": norm, "Material name": name})

    g = cur.groupby("new_master_code", sort=False)
    master = g[["norm_material_name", "Material name", "_sim"]].apply(canonical)
    master["cluster_id"] = g["cluster_id"].first()
    master["cluster_record_count"] = g["cluster_record_count"].first()
    master["Category"] = category_for(slug)
    master = master.reset_index()[FINAL_COLS]
    return cur, master

In [ ]:
unique_masters = {}
for f in curated_files:
    slug = f.stem.replace("curated_", "", 1)
    cur, master = build_unique_master(slug)

    assert master["new_master_code"].is_unique, f"{slug}: duplicate master codes"
    assert master["cluster_record_count"].astype(int).sum() == len(cur), f"{slug}: row count mismatch"

    out_path = UNIQUE_DIR / f"unique_master_{slug}.csv"
    master.to_csv(out_path, index=False)
    unique_masters[slug] = master
    print(f"[{slug}] curated rows read: {len(cur):,} | unique master rows: {len(master):,} | saved: {out_path}")

In [ ]:
# Quick look (plots only; nothing is saved or merged on disk)
import matplotlib.pyplot as plt
counts = pd.Series({s: len(m) for s, m in unique_masters.items()})
ax = counts.plot.bar(color="#4C72B0", figsize=(8, 4), title="Unique master materials per category")
ax.bar_label(ax.containers[0]); plt.tight_layout(); plt.show()
display(next(iter(unique_masters.values())).head(10))